# Preprocess Vietnamese News Captions

This notebook standardizes image captions in the Vietnamese news dataset by removing photographer/source attribution templates (e.g., ` - Ảnh: VGP/Vũ Phong`).


In [10]:
import json
import re
import unicodedata
from pathlib import Path
from collections import Counter

## 1. Load Dataset


In [11]:
# Load the JSON dataset
input_path = Path("../data/json/news_data_vifactcheck_test.json")
output_path = Path("../data/json/news_data_vifactcheck_test_cleaned.json")

with open(input_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data)} articles")

Loaded 1282 articles


## 2. Analyze Caption Patterns


In [12]:
# Collect all captions
all_captions = []
for article in data:
    for img in article.get("images", []):
        caption = img.get("caption", "")
        if caption:
            all_captions.append(caption)

print(f"Total captions: {len(all_captions)}")

# Find captions with dash patterns (potential attribution)
dash_captions = [c for c in all_captions if " - " in c]
print(f"Captions with ' - ': {len(dash_captions)}")

# Show some examples
print("\nSample captions with dash:")
for c in dash_captions[:10]:
    print(f"  - {c[:100]}...")

Total captions: 1656
Captions with ' - ': 503

Sample captions with dash:
  - Hệ thống điện tại Côn Đảo hiện nay được cấp chủ yếu từ nguồn điện chạy dầu - Ảnh: N.HIỂN...
  - Người Nga tới nhà ga ở Helsinki, Phần Lan, được chào đón bằng những tấm biển ghi "Chào mừng" và "Hòa...
  - Diễn đàn "Thúc đẩy số hóa trong truy xuất nguồn gốc nông sản - thực phẩm" diễn ra trực tuyến lẫn trự...
  - Một sản phẩm nông sản khi đăng ký truy xuất nguồn gốc cần theo dõi từ lúc ở trang trại cho tới khi đ...
  - Chia ô Đảng kỳ (chấm “xanh” là tâm Búa - Liềm)...
  - Đại diện viện kiểm sát trình bày quan điểm giải quyết án phúc thẩm liên quan vụ bán rẻ hai dự án Phư...
  - Thủ tướng Singapore Lý Hiển Long phát biểu tại Diễn đàn châu Á Bác Ngao ngày 30-3 ở Trung Quốc - Ảnh...
  - Tòa biệt thự số 45 đường Barker - Ảnh: GOOGLE MAPS...
  - Các bất động sản tư nhân ở khu vực The Peak của Hong Kong - Ảnh: BLOOMBERG...
  - Ông Hoàng Văn Quân (phóng viên Hoàng Quân) trình báo sự việc cả gia đình ông bị gọi điện dọa

## 3. Define Attribution Patterns


In [13]:
# Regex patterns for Vietnamese attribution removal
# Pattern: text followed by dash and attribution keywords
ATTRIBUTION_PATTERNS = [
    # Vietnamese patterns: Ảnh:, Nguồn:, etc.
    r"\s*[-–.—]\s*Ảnh\s*:.*$",
    r"\s*[-–.—]\s*Ảnh\s+minh\s+họa.*$",  # Ảnh minh họa (no colon)
    r"\s*[-–.—]\s*Nguồn\s*:.*$",
    r"\s*[-–.—]\s*Photo\s*:.*$",
    r"\s*[-–.—]\s*Image\s*:.*$",
    r"\s*[-–.—]\s*Courtesy\s*:.*$",
    r"\s*[-–.—]\s*Copyright\s*:.*$",
    r"\s*[-–.—]\s*©.*$",
    r"\s*[-–.—]\s*VGP\s*/.*$",
    r"\s*[-–.—]\s*VNA.*$",
    r"\s*[-–.—]\s*Đồ\s*hoạ\s*:.*$",  # Đồ hoạ:
    r"\s*[-–.—]\s*Đồ\s*họa\s*:.*$",  # Đồ họa:
    r"\s*\(Ảnh\s*:[^)]+\)\s*[.]*\s*$",  # (Ảnh: ...) with optional trailing period
]


def clean_caption(caption):
    """Remove attribution templates from caption."""
    if not caption or not isinstance(caption, str):
        return caption

    # Normalize Unicode to handle different representations (e.g., Ả vs Ả)
    cleaned = unicodedata.normalize("NFC", caption)

    for pattern in ATTRIBUTION_PATTERNS:
        cleaned = re.sub(pattern, "", cleaned, flags=re.IGNORECASE)

    # Clean up trailing whitespace
    cleaned = cleaned.strip()

    return cleaned

## 4. Test Cleaning Function


In [14]:
# Test with sample captions
test_captions = [
    "Trưởng Ban Tuyên giáo Trung ương Nguyễn Trọng Nghĩa phát biểu - Ảnh: VGP/Vũ Phong",
    "Tổng Biên tập Báo Nhân dân Lê Quốc Minh trình bày báo cáo - Ảnh: VGP/Vũ Phong",
    "Hình ảnh khách sạn những ngày đầu mới thành lập",
    "Phó chủ tịch UBND tỉnh trao cờ đơn vị dẫn đầu - Nguồn: thanhnien.vn",
]

print("Before vs After:")
for caption in test_captions:
    cleaned = clean_caption(caption)
    print(f"  BEFORE: {caption}")
    print(f"  AFTER:  {cleaned}")
    print()

Before vs After:
  BEFORE: Trưởng Ban Tuyên giáo Trung ương Nguyễn Trọng Nghĩa phát biểu - Ảnh: VGP/Vũ Phong
  AFTER:  Trưởng Ban Tuyên giáo Trung ương Nguyễn Trọng Nghĩa phát biểu

  BEFORE: Tổng Biên tập Báo Nhân dân Lê Quốc Minh trình bày báo cáo - Ảnh: VGP/Vũ Phong
  AFTER:  Tổng Biên tập Báo Nhân dân Lê Quốc Minh trình bày báo cáo

  BEFORE: Hình ảnh khách sạn những ngày đầu mới thành lập
  AFTER:  Hình ảnh khách sạn những ngày đầu mới thành lập

  BEFORE: Phó chủ tịch UBND tỉnh trao cờ đơn vị dẫn đầu - Nguồn: thanhnien.vn
  AFTER:  Phó chủ tịch UBND tỉnh trao cờ đơn vị dẫn đầu



## 5. Process All Captions


In [15]:
# Process all articles
cleaned_count = 0
total_captions = 0

for article in data:
    for img in article.get("images", []):
        caption = img.get("caption", "")
        if caption:
            total_captions += 1
            cleaned = clean_caption(caption)
            if cleaned != caption:
                cleaned_count += 1
            img["caption"] = cleaned

print(f"Processed {total_captions} captions")
print(f"Modified {cleaned_count} captions ({cleaned_count/total_captions*100:.1f}%)")

Processed 1656 captions
Modified 1023 captions (61.8%)


## 6. Save Cleaned Dataset


In [16]:
# Ensure output directory exists
output_path.parent.mkdir(parents=True, exist_ok=True)

# Save cleaned data
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

print(f"Saved cleaned dataset to: {output_path}")
print(f"File size: {output_path.stat().st_size / 1024:.1f} KB")

Saved cleaned dataset to: ../data/json/news_data_vifactcheck_test_cleaned.json
File size: 9511.5 KB


## 7. Validation


In [17]:
# Reload and verify
with open(output_path, "r", encoding="utf-8") as f:
    verified_data = json.load(f)

print(f"Verified: {len(verified_data)} articles loaded")

# Check for remaining attribution patterns
remaining = []
for article in verified_data:
    for img in article.get("images", []):
        caption = img.get("caption", "")
        if caption and " - " in caption:
            remaining.append(caption)

print(f"\nCaptions still containing ' - ': {len(remaining)}")
print("\nAll remaining captions with dash:")
for i, c in enumerate(remaining, 1):
    print(f"{i}. {c}")

Verified: 1282 articles loaded

Captions still containing ' - ': 84

All remaining captions with dash:
1. Diễn đàn "Thúc đẩy số hóa trong truy xuất nguồn gốc nông sản - thực phẩm" diễn ra trực tuyến lẫn trực tiếp, ở TP.HCM và Hà Nội. Trong hình: Đại diện Bộ Nông nghiệp và Phát triển nông thôn cùng các doanh nghiệp tham gia diễn đàn tại Hà Nội
2. Chia ô Đảng kỳ (chấm “xanh” là tâm Búa - Liềm)
3. Đương kim Hoa hậu Thế giới Karolina Bielawska và Chủ tịch cuộc thi Hoa hậu Thế giới - Julia Morley được truyền thông Ấn Độ chào đón
4. Phát triển du lịch Hà Nội đảm bảo bền vững, hiệu quả, vừa là cửa ngõ đón và phân phối khách du lịch ở phía bắc và cả nước; vừa là điểm đến du lịch "An toàn - Thân thiện - Chất lượng - Hấp dẫn"
5. Nghệ sĩ Võ Minh Lâm trong chương trình Vầng trăng cổ nhạc truyền hình trực tiếp trên kênh HTV9 tối 26-3 - Ảnh chụp màn hình: LINH ĐOAN
6. Bà Nguyễn Thị Thanh - nạn nhân sống sót trong vụ thảm sát của lính Hàn Quốc tại làng Phong Nhất và Phong Nhị ngày 12/2/1968
7. 33.509